In [15]:
!pip install pyproj

In [21]:
import os
import cv2
import numpy as np
import tensorflow as tf
from ultralytics import YOLO
from pyproj import Geod, Proj
import re
import math

# --- НАЛАШТУВАННЯ ---
YOLO_MODEL_PATH = "runs/detect/yolo_only/weights/best.pt"
CNN_MODEL_PATH = "radar_digit_classifier.keras"

USE_CNN_REFINEMENT = False

TEST_IMAGES_DIR = "final_dataset_yoloOnly/images/test"
OUTPUT_DIR = "pipeline_results"

RADAR_LAT = 48.5
RADAR_LON = 35.1

# Поріг відстані для дублікатів (пікселі)
DUPLICATE_DIST_THRESH = 5.0
# Поріг вертикального відхилення для рядка (відносно висоти символу)
LINE_VERTICAL_TOLERANCE = 0.8
# Поріг горизонтального розриву (новий блок даних)
BLOCK_GAP_MULTIPLIER = 2.5

YOLO_NAMES = {
    0: '0', 1: '1', 2: '2', 3: '3', 4: '4',
    5: '5', 6: '6', 7: '7', 8: '8', 9: '9',
    10: 'minus', 11: 'point', 12: 'tag'
}

CHAR_MAP = {
    '0': '0', '1': '1', '2': '2', '3': '3', '4': '4',
    '5': '5', '6': '6', '7': '7', '8': '8', '9': '9',
    'minus': '-', 'point': '.', 'tag': ' '
}

CNN_NAMES = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'minus', 'point', 'tag']

os.makedirs(OUTPUT_DIR, exist_ok=True)

def preprocess_for_cnn(img_crop):
    if len(img_crop.shape) == 3:
        img_crop = cv2.cvtColor(img_crop, cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img_crop, (32, 32))
    img = img.astype("float32") / 255.0
    img = np.expand_dims(img, axis=-1)
    img = np.expand_dims(img, axis=0)
    return img

def get_cnn_class(cnn_model, crop):
    try:
        input_tensor = preprocess_for_cnn(crop)
        preds = cnn_model.predict(input_tensor, verbose=0)
        idx = np.argmax(preds)
        return CNN_NAMES[idx]
    except:
        return "?"

def filter_duplicates(chars):
    """Видаляє дублікати (наприклад, дві точки одна на одній)"""
    if not chars: return []

    # Сортуємо за впевненістю (найвпевненіші перші)
    chars.sort(key=lambda x: x['conf'], reverse=True)

    valid_chars = []
    for c in chars:
        is_duplicate = False
        for valid in valid_chars:
            # Евклідова відстань між центрами
            dist = math.hypot(c['cx'] - valid['cx'], c['cy'] - valid['cy'])
            if dist < DUPLICATE_DIST_THRESH:
                is_duplicate = True
                break
        if not is_duplicate:
            valid_chars.append(c)

    return valid_chars

def organize_text_lines(chars):
    """
    Розумне сортування символів у рядки за допомогою векторів.
    Будує ланцюжки зліва направо, враховуючи кут нахилу.
    """
    if not chars: return ""

    # 1. Фільтруємо дублікати
    chars = filter_duplicates(chars)

    # 2. Сортуємо всі символи строго зліва направо (по X)
    # Це база для побудови ланцюжка.
    chars.sort(key=lambda c: c['cx'])

    lines = [] # Список списків (кожен список - це рядок символів)

    # Налаштування допусків
    MAX_ANGLE_DEG = 25.0  # Максимальний нахил рядка (градуси)
    MAX_GAP_RATIO = 2.5   # Максимальний розрив між буквами (в ширинах букви)

    for char in chars:
        best_line = None
        min_dist = float('inf')

        # Шукаємо, до якого рядка можна причепити цей символ
        for line in lines:
            last = line[-1] # Останній символ цього рядка

            # Вектор від останнього до поточного
            dx = char['cx'] - last['cx']
            dy = char['cy'] - last['cy']

            # Символ має бути справа (dx > 0).
            # Допускаємо мізерний від'ємний зсув (-5px) тільки якщо це вертикальний стек,
            # але тут ми будуємо рядки, тому dx має бути суттєвим.
            if dx <= 0: continue

            dist = math.hypot(dx, dy)

            # 1. Перевірка кута (щоб не стрибати на сусідні рядки)
            angle = math.degrees(math.atan2(dy, dx))
            if abs(angle) > MAX_ANGLE_DEG: continue

            # 2. Перевірка відстані (щоб не об'єднувати слова через пів екрану)
            # Беремо середній розмір символів як еталон
            avg_size = (last['w'] + char['w'] + last['h'] + char['h']) / 4.0
            max_allowed_dist = avg_size * MAX_GAP_RATIO * 2.0 # Трохи з запасом

            if dist > max_allowed_dist: continue

            # Якщо підходить, перевіряємо, чи це найближчий варіант
            if dist < min_dist:
                min_dist = dist
                best_line = line

        if best_line:
            best_line.append(char)
        else:
            # Не підійшов до жодного рядка -> створюємо новий
            lines.append([char])

    # 3. Сортуємо самі рядки зверху вниз (за середнім Y)
    # Це щоб X: ... йшло перед Y: ...
    lines.sort(key=lambda l: sum(c['cy'] for c in l) / len(l))

    # 4. Збираємо текст
    full_text = ""
    for line in lines:
        line_str = ""
        prev_x = line[0]['x'] + line[0]['w']

        for i, char_obj in enumerate(line):
            # Додаємо пробіл, якщо є візуальний розрив всередині рядка
            if i > 0:
                curr_x = char_obj['x']
                avg_w = (char_obj['w'] + line[i-1]['w']) / 2
                gap = curr_x - prev_x
                if gap > avg_w * 0.8: # Якщо дірка більша за 80% ширини літери
                    line_str += " "

            line_str += char_obj['char']
            prev_x = char_obj['x'] + char_obj['w']

        full_text += line_str + " " # Пробіл між рядками

    return full_text.strip()

def convert_coordinates(text_data):
    clean_text = re.sub(r'\s+', ' ', text_data).strip()

    # 1. Спроба знайти мітки
    az_match = re.search(r'(?:Az|A)[:\s]+(-?[\d\.]+)', clean_text, re.IGNORECASE)
    rg_match = re.search(r'(?:Rg|R)[:\s]+(-?[\d\.]+)', clean_text, re.IGNORECASE)
    if az_match and rg_match:
        try:
            az = float(az_match.group(1))
            dist = float(rg_match.group(1))
            geod = Geod(ellps='WGS84')
            lon_end, lat_end, _ = geod.fwd(RADAR_LON, RADAR_LAT, az, dist)
            return f"GPS: {lat_end:.5f}, {lon_end:.5f} (Polar)"
        except: pass

    x_match = re.search(r'X[:\s]+(-?[\d\.]+)', clean_text, re.IGNORECASE)
    y_match = re.search(r'Y[:\s]+(-?[\d\.]+)', clean_text, re.IGNORECASE)
    if x_match and y_match:
        try:
            x = float(x_match.group(1))
            y = float(y_match.group(1))
            proj_str = f"+proj=ortho +lat_0={RADAR_LAT} +lon_0={RADAR_LON} +datum=WGS84"
            p = Proj(proj_str)
            lon_end, lat_end = p(x, y, inverse=True)
            return f"GPS: {lat_end:.5f}, {lon_end:.5f} (Cartesian)"
        except: pass

    # 2. Фолбек: Шукаємо просто пару чисел
    numbers = re.findall(r'-?[\d\.]+', clean_text)
    # Фільтруємо короткі "числа" які можуть бути сміттям (напр. поодинокі '.')
    numbers = [n for n in numbers if len(n) > 1 or n.isdigit()]

    if len(numbers) >= 2:
        try:
            # Беремо перші два нормальні числа
            val1 = float(numbers[0])
            val2 = float(numbers[1])

            # Припускаємо Polar, бо це найчастіше
            geod = Geod(ellps='WGS84')
            lon_end, lat_end, _ = geod.fwd(RADAR_LON, RADAR_LAT, val1, val2)
            return f"GPS: {lat_end:.5f}, {lon_end:.5f} (Auto-Polar)"
        except: pass

    return f"Raw: {clean_text}"

def main():
    print(f"Loading YOLO from {YOLO_MODEL_PATH}...")
    try:
        yolo_model = YOLO(YOLO_MODEL_PATH)
    except:
        print("❌ YOLO модель не знайдено.")
        return

    cnn_model = None
    if USE_CNN_REFINEMENT:
        print(f"Loading CNN from {CNN_MODEL_PATH}...")
        try:
            cnn_model = tf.keras.models.load_model(CNN_MODEL_PATH)
        except:
            print("⚠️ CNN не знайдено! Працюємо тільки на YOLO.")

    test_files = [os.path.join(TEST_IMAGES_DIR, f) for f in os.listdir(TEST_IMAGES_DIR)
                  if f.endswith(('.jpg', '.png'))][:15]

    print(f"Processing {len(test_files)} images...")

    for img_path in test_files:
        img = cv2.imread(img_path)
        if img is None: continue
        vis_img = img.copy()

        # Знизив поріг, щоб ловити навіть тьмяні символи
        results = yolo_model(img_path, conf=0.20, verbose=False)

        all_detected_chars = []

        for r in results:
            boxes = r.boxes
            for box in boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])

                h, w = img.shape[:2]
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)

                yolo_label = YOLO_NAMES.get(int(box.cls[0]), '?')
                final_label = yolo_label

                if cnn_model:
                    crop = img[y1:y2, x1:x2]
                    if crop.size > 0:
                        cnn_label = get_cnn_class(cnn_model, crop)
                        final_label = cnn_label # CNN авторитетніша для форми

                final_char_symbol = CHAR_MAP.get(final_label, '?')

                all_detected_chars.append({
                    'char': final_char_symbol,
                    'cx': (x1+x2)/2, 'cy': (y1+y2)/2,
                    'x': x1, 'y': y1, 'w': x2-x1, 'h': y2-y1,
                    'conf': conf
                })

                color = (0, 255, 0)
                if final_label == 'point': color = (0, 255, 255)
                cv2.rectangle(vis_img, (x1, y1), (x2, y2), color, 1)

        # РОЗУМНА ЗБІРКА ТЕКСТУ
        full_text = organize_text_lines(all_detected_chars)

        # Конвертація
        gps_res = convert_coordinates(full_text)
        print(f"[{os.path.basename(img_path)}] '{full_text}' -> {gps_res}")

        cv2.putText(vis_img, gps_res, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
        cv2.putText(vis_img, f"Raw: {full_text}", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)

        cv2.imwrite(os.path.join(OUTPUT_DIR, os.path.basename(img_path)), vis_img)

    print(f"✅ Результати в {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

Loading YOLO from runs/detect/yolo_only/weights/best.pt...
Processing 15 images...
[000000_v0_aug_12.jpg] '22.2    4456.7' -> GPS: 48.53710, 35.12280 (Auto-Polar)
[000000_v0_aug_7.jpg] '22.2    4456.7' -> GPS: 48.53710, 35.12280 (Auto-Polar)
[000000_v0_aug_9.jpg] '22.2    4456.7' -> GPS: 48.53710, 35.12280 (Auto-Polar)
[000000_v1_aug_4.jpg] '7570   8209   1484' -> GPS: 48.57270, 35.11932 (Auto-Polar)
[000000_v1_aug_7.jpg] '7570   8209   1484' -> GPS: 48.57270, 35.11932 (Auto-Polar)
[000000_v2_aug_3.jpg] '7453   -6190  2455' -> GPS: 48.51625, 35.18013 (Auto-Polar)
[000001_v0.jpg] '100.8    4005.2' -> GPS: 48.49324, 35.15323 (Auto-Polar)
[000001_v0_aug_8.jpg] '100.8    4005.2' -> GPS: 48.49324, 35.15323 (Auto-Polar)
[000001_v0_aug_9.jpg] '100.8    4005.2' -> GPS: 48.49324, 35.15323 (Auto-Polar)
[000001_v1.jpg] '449183.09    5234921.08' -> GPS: 26.84482, -19.35891 (Auto-Polar)
[000001_v1_aug_10.jpg] '449183.09   5234921.08' -> GPS: 26.84482, -19.35891 (Auto-Polar)
[000001_v1_aug_12.jpg] '